<a href="https://colab.research.google.com/github/Phreely/Boltz-2_YAML_generator/blob/main/Boltz_2_yaml_advanced.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title Input protein sequence(s), then hit `Runtime` -> `Run all`
#@markdown use ´:´ as separator for multiple inputs
from google.colab import files
import os
import re
import hashlib
import requests
import yaml
import json
from string import ascii_uppercase

# User inputs
query_sequence = ''  #@param {type:"string"}
ligand_input_smiles = ''  #@param {type:"string"}
ligand_input_ccd = ''  #@param {type:"string"}
ligand_input_common_name = ''  #@param {type:"string"}
dna_input = ''  #@param {type:"string"}
jobname = ''  #@param {type:"string"}

#@markdown ---
#@markdown ### **Advanced Settings**
#@markdown Enter a number for `max_msa`. Leave at 0 to use Boltz-2 defaults.
max_msa = 0 #@param {type:"integer"}
recycling_steps = "auto" #@param ["auto", "1", "3", "5", "10"]

#@markdown ---
#@markdown ### **Constraints Input**
#@markdown Leave fields blank to ignore a constraint type.
#@markdown
#@markdown **Atom Formatting: `[ChainID], [ResidueNumber], [AtomName]`** (e.g., `B, 64, NZ` or `LC, 1, C20`)
#@markdown
#@markdown #### **1. Bond Constraint**
bond_atom1 = "" #@param {type:"string"}
bond_atom2 = "" #@param {type:"string"}
#@markdown #### **2. Contact Constraint**
contact_token1 = "" #@param {type:"string"}
contact_token2 = "" #@param {type:"string"}
contact_distance = 6.0 #@param {type:"number"}
contact_force = False #@param {type:"boolean"}
#@markdown #### **3. Pocket Constraint**
pocket_binder = "" #@param {type:"string"}
pocket_contact1 = "" #@param {type:"string"}
pocket_contact2 = "" #@param {type:"string"}
pocket_distance = 6.0 #@param {type:"number"}
pocket_force = False #@param {type:"boolean"}

#@markdown ---
#@markdown ### **Templates**
#@markdown Specify which protein chain should be modelled using a template and provide the template PDB ID.
template_chain = "" #@param {type:"string"}
template_pdb_id = "Insert link to PDB or CIF file here" #@param {type:"string"}

#@markdown ---
#@markdown ### **Affinity Prediction**
#@markdown Specify a ligand chain ID (e.g., `LB`, `CC`) to compute its binding affinity.
compute_affinity_ligand = "" #@param {type:"string"}

# 1. Clean up and Uppercase
query_sequence = re.sub(r'\s+', '', query_sequence).upper()
dna_input = re.sub(r'\s+', '', dna_input).upper()
ligand_input_smiles = re.sub(r'\s+', '', ligand_input_smiles) # SMILES are case-sensitive
ligand_input_ccd = re.sub(r'\s+', '', ligand_input_ccd).upper()
ligand_input_common_name = re.sub(r'\s+', '', ligand_input_common_name)

# 2. Setup Jobname and Directory
basejobname = re.sub(r'\W+', '', jobname)
jobname = basejobname + "_" + hashlib.sha1(query_sequence.encode()).hexdigest()[:5]
os.makedirs(jobname, exist_ok=True)

# 3. Handle Common Names via PubChem
def get_smiles(compound_name):
    try:
        url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{compound_name}/property/CanonicalSMILES/JSON"
        r = requests.get(url, timeout=5)
        return r.json()['PropertyTable']['Properties'][0]['CanonicalSMILES']
    except:
        return None

protein_sequences = query_sequence.split(':') if query_sequence else []
dna_sequences = dna_input.split(':') if dna_input else []
smiles_ligands = ligand_input_smiles.split(':') if ligand_input_smiles else []
ccd_ligands = ligand_input_ccd.split(':') if ligand_input_ccd else []

if ligand_input_common_name:
    for name in ligand_input_common_name.split(':'):
        smi = get_smiles(name)
        if smi:
            print(f"Found SMILES for {name}: {smi}")
            smiles_ligands.append(smi)

# 4. Construct YAML Dictionary
boltz_dict = {
    "name": jobname,
    "sequences": []
}

chain_gen = iter(ascii_uppercase)

# Add Proteins
for seq in protein_sequences:
    if seq:
        chain_id = next(chain_gen)
        prot_entry = {"protein": {"id": chain_id, "sequence": seq}}
        if template_chain and template_pdb_id and chain_id == template_chain.upper():
            prot_entry["protein"]["templates"] = [template_pdb_id]
        boltz_dict["sequences"].append(prot_entry)

# Add DNA
for seq in dna_sequences:
    if seq:
        boltz_dict["sequences"].append({"dna": {"id": next(chain_gen), "sequence": seq}})

# Add SMILES Ligands
for i, smi in enumerate(smiles_ligands):
    if smi:
        boltz_dict["sequences"].append({"ligand": {"id": f"L{next(chain_gen)}", "smiles": smi}})

# Add CCD Ligands
for ccd in ccd_ligands:
    if ccd:
        boltz_dict["sequences"].append({"ligand": {"id": f"C{next(chain_gen)}", "ccd": ccd}})

# 5. Advanced Parameters
sampling = {}
if recycling_steps != "auto":
    sampling["recycling_steps"] = int(recycling_steps)
if sampling:
    boltz_dict["sampling"] = sampling

if max_msa > 0:
    boltz_dict["msa"] = {"max_msa_seqs": max_msa}

# Setup custom FlowList to format YAML arrays inline
class FlowList(list):
    pass

def represent_flow_list(dumper, data):
    return dumper.represent_sequence('tag:yaml.org,2002:seq', data, flow_style=True)

yaml.add_representer(FlowList, represent_flow_list)

# Parse and format constraints properly
def parse_atom_list(atom_str):
    parts = [p.strip() for p in atom_str.split(',')]
    parsed = FlowList()
    for p in parts:
        if p.isdigit():
            parsed.append(int(p))
        else:
            parsed.append(p)
    return parsed

formatted_constraints = []

if bond_atom1.strip() and bond_atom2.strip():
    try:
        formatted_constraints.append({
            "bond": {
                "atom1": parse_atom_list(bond_atom1),
                "atom2": parse_atom_list(bond_atom2)
            }
        })
    except Exception as e:
        print(f"Warning: Could not parse bond constraints correctly. Error: {e}")

if contact_token1.strip() and contact_token2.strip():
    try:
        formatted_constraints.append({
            "contact": {
                "token1": parse_atom_list(contact_token1),
                "token2": parse_atom_list(contact_token2),
                "max_distance": float(contact_distance),
                "force": bool(contact_force)
            }
        })
    except Exception as e:
        print(f"Warning: Could not parse contact constraints correctly. Error: {e}")

if pocket_binder.strip() and (pocket_contact1.strip() or pocket_contact2.strip()):
    try:
        binder_parsed = parse_atom_list(pocket_binder)
        binder_val = binder_parsed[0] if len(binder_parsed) == 1 else binder_parsed

        contacts = []
        if pocket_contact1.strip():
            contacts.append(parse_atom_list(pocket_contact1))
        if pocket_contact2.strip():
            contacts.append(parse_atom_list(pocket_contact2))

        formatted_constraints.append({
            "pocket": {
                "binder": binder_val,
                "contacts": contacts,
                "max_distance": float(pocket_distance),
                "force": bool(pocket_force)
            }
        })
    except Exception as e:
        print(f"Warning: Could not parse pocket constraints correctly. Error: {e}")

if formatted_constraints:
    boltz_dict["constraints"] = formatted_constraints

# Add affinity properties if provided
if compute_affinity_ligand.strip():
    boltz_dict["properties"] = [{"affinity": {"binder": compute_affinity_ligand.strip()}}]

# 6. Save and Download
yaml_path = os.path.join(jobname, f"{jobname}.yaml")
with open(yaml_path, 'w') as f:
    yaml.dump(boltz_dict, f, default_flow_style=False, sort_keys=False)

print(f"\nSuccess! YAML created at {yaml_path}")
files.download(yaml_path)



Success! YAML created at HydF_Hmet_b0589/HydF_Hmet_b0589.yaml


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>